# Solution: Tree-Based Models

<div style="text-align: right; font-size: 1.2em; color: #333;">
This part is created by Xueqin(Ned) Chen
</div>

Tree-based models are a type of machine learning algorithm that builds a decision tree structure to make predictions or decisions based on input features. These models are commonly used for both classification and regression tasks. The decision tree is constructed by recursively partitioning the input space into subsets, making decisions at each internal node based on feature values. Each leaf node represents the final prediction or outcome. Popular examples of tree-based models include Decision Trees, Random Forests, Gradient Boosted Trees, and XGBoost. They are valued for their interpretability, versatility, and effectiveness in capturing complex relationships in data.</p>
In this Jupyter Notebook, we will delve into the application of the <code>scikit-learn</code> library to build a decision tree and tackle a classification problem, specifically <strong>rain prediction in Australia</strong>. The approach involves creating a basic decision tree and subsequently enhancing its performance through the utilization of ensemble methods.</p>

<div style="flex: 30%; padding: 20px;">
    <img src="https://miro.medium.com/v2/resize:fit:522/format:webp/1*-lZIWa8PuUO9iSMtOCn1xQ.png" alt="Tree Image" style="display: block; margin: auto; max-width: 100%;">
    <p style="text-align: center; margin-top: 10px;">Decision Tree. Icons made by <a href="https://www.flaticon.com/authors/becris">Becris</a> from Flaticon.</p>
</div>

# 1. Import modules

In [ ]:
import warnings
warnings.filterwarnings('ignore') # this code can help you ignore some warning messages

import pandas as pd
import numpy as np
import random
import os
from urllib.request import urlretrieve

%matplotlib inline 
# the matplotlib inline is a jupyter notebook specific command that let's you see the plots in the notebook itself.
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
from sklearn import preprocessing
from sklearn import metrics
import sklearn as sk

# these codes can help you fix the random seed so that you can get the same result each time you run the code
random.seed(42)
np.random.seed(42)
sk.random.seed(42)

# 2. Dataset

<p style="line-height: 1.6;">
    In this experiment, we will use the <strong>Australia Weather Data</strong>. You can find more description in the following link:
</p>

<p>
    <a href="https://www.kaggle.com/datasets/arunavakrchakraborty/australia-weather-data" >Australia Weather Data | Kaggle</a>
</p>

<div>
    Here, we provide some basic knowledge of each column in the dataset.
    <table style="border-collapse: collapse; table-layout: auto;" border="1">
        <tr>
            <th> No. </th>
            <th>Feature Name</th>
            <th>Description</th>
        </tr>
        <tr>
            <td>1</td>
            <td>Location</td>
            <td>Name of the city from Australia.</td>
        </tr>
        <tr>
            <td>2</td>
            <td>MinTemp</td>
            <td>The Minimum temperature during a particular day. (degree Celsius)</td>
        </tr>
        <tr>
            <td>3</td>
            <td>MaxTemp</td>
            <td>The maximum temperature during a particular day. (degree Celsius)</td>
        </tr>
        <tr>
            <td>4</td>
            <td>Rainfall</td>
            <td>Rainfall during a particular day. (millimeters)</td>
        </tr>
        <tr>
            <td>5</td>
            <td>Evaporation</td>
            <td>Evaporation during a particular day. (millimeters)</td>
        </tr>
        <tr>
            <td>6</td>
            <td>Sunshine</td>
            <td>Bright sunshine during a particular day. (hours)</td>
        </tr>
        <tr>
            <td>7</td>
            <td>WindGustDir</td>
            <td>The direction of the strongest gust during a particular day. (16 compass points)</td>
        </tr>
        <tr>
            <td>8</td>
            <td>WindGustSpeed</td>
            <td>Speed of strongest gust during a particular day. (kilometers per hour)</td>
        </tr>
        <tr>
            <td>9</td>
            <td>WindDir9am</td>
            <td>The direction of the wind for 10 min prior to 9 am. (compass points)</td>
        </tr>
        <tr>
            <td>10</td>
            <td>WindDir3pm</td>
            <td>The direction of the wind for 10 min prior to 3 pm. (compass points)</td>
        </tr>
        <tr>
            <td>11</td>
            <td>WindSpeed9am</td>
            <td>Speed of the wind for 10 min prior to 9 am. (kilometers per hour)</td>
        </tr>
        <tr>
            <td>12</td>
            <td>WindSpeed3pm</td>
            <td>Speed of the wind for 10 min prior to 3 pm. (kilometers per hour)</td>
        </tr>
        <tr>
            <td>13</td>
            <td>Humidity9am</td>
            <td>The humidity of the wind at 9 am. (percent)</td>
        </tr>
        <tr>
            <td>14</td>
            <td>Humidity3pm</td>
            <td>The humidity of the wind at 3 pm. (percent)</td>
        </tr>
        <tr>
            <td>15</td>
            <td>Pressure9am</td>
            <td>Atmospheric pressure at 9 am. (hectopascals)</td>
        </tr>
        <tr>
            <td>16</td>
            <td>Pressure3pm</td>
            <td>Atmospheric pressure at 3 pm. (hectopascals)</td>
        </tr>
        <tr>
            <td>17</td>
            <td>Cloud9am</td>
            <td>Cloud-obscured portions of the sky at 9 am. (eighths)</td>
        </tr>
        <tr>
            <td>18</td>
            <td>Cloud3pm</td>
            <td>Cloud-obscured portions of the sky at 3 pm. (eighths)</td>
        </tr>
        <tr>
            <td>19</td>
            <td>Temp9am</td>
            <td>The temperature at 9 am. (degree Celsius)</td>
        </tr>
        <tr>
            <td>20</td>
            <td>Temp3pm</td>
            <td>The temperature at 3 pm. (degree Celsius)</td>
        </tr>
        <tr>
            <td>21</td>
            <td>RainToday</td>
            <td>If today is rainy then ‘Yes’. If today is not rainy then ‘No’.</td>
        </tr>
        <tr>
            <td>22</td>
            <td>RainTomorrow</td>
            <td>If tomorrow is rainy then 1 (Yes). If tomorrow is not rainy then 0 (No).</td>
        </tr>      
    </table>
</div>

## 2.1 Download and load dataset

In [ ]:
train_file = 'Weather Training Data.csv'

# Download the Australia weather data (if necessary)
train_url = "https://surfdrive.surf.nl/files/index.php/s/a4HmL6NWnEn3C8W/download"
test_url = "https://surfdrive.surf.nl/files/index.php/s/Kg5NFKSgd3qRjvO/download"

if not os.path.isfile(train_file):
    print("Downloading Australia weather dataset...")
    urlretrieve(train_url, "Weather Training Data.csv")
    urlretrieve(test_url, "Weather Test Data.csv")

# load training data
data_train = pd.read_csv('Weather Training Data.csv')

# show data information
data_train.head()

You also can load the test data...

In [ ]:
# load test data
data_test = pd.read_csv('Weather Test Data.csv')
# show data information
data_test.head()

NOTE! Because the **Weather Test Data.csv** do not have ground truth, in this notebook, we only use **Weather Training Data.csv** for both training and testing!

## 2.2 Preprocessing

### 2.2.1 Dealing with missing data

First, we need to identify the missing data. We can do it separately for training and testing datasets, or concat two datasets and processing together. In our notebook, we chose the first method, and you guys can use the second one to test which one is better!

```python
# we use pd.concat to stack two datasets.
df=pd.concat([data_train, data_test], ignore_index=True, sort=False)
df.head()
# we can use df.shape to see how many rows and columns in the new dataset.
df.shape
```

In [ ]:
# Then use df.info to see the details of dataset.
data_train.info()

<p>We can check for missing values in a DataFrame named <code>data_train</code> by using the <code>isna()</code> function from the <code>pandas</code> library in Python. </p>
<p>This function will return a DataFrame of the same size as <code>data_train</code> but with True in the locations where <code>data_train</code> has NaN or None and False elsewhere.</p>
 

In [ ]:
missing = data_train.isnull()
missing.head()

<p>Then we can use <code>missing.sum()</code> to simply statistic the null value in each column.</p>

In [ ]:
# this command can simply statistic the null value in each column.
missing.sum()

To deal with missing data, we should use different methods to cope with different types of data. We first look at the continuous data. Here are several simple strategies for dealing with missing data in continuous variables:

1. *Remove Rows with Missing Data*: one straightforward approach is to remove rows (samples) that contain missing data for the continuous variable of interest. This is a simple solution but may lead to a loss of valuable information if you have a small dataset.
```python
# Remove rows with missing data for a specific column
df.dropna(subset=['Column_Name'], inplace=True)
```
2. *Impute with Mean/Median*: we can impute missing values with the mean or median of the continuous variable. This helps to maintain the dataset's overall statistical properties.
```python
# Impute missing values with the mean of the column
df['Column_Name'].fillna(df['Column_Name'].mean(), inplace=True)
# Impute missing values with the median of the column
df['Column_Name'].fillna(df['Column_Name'].median(), inplace=True)
```
3. *Impute with a Specific Value*: we can impute missing values with a specific value that makes sense in the context of your data. For example, you might use zero, a negative value, or another constant.
```python
# Impute missing values with a specific value (e.g., 0)
df['Column_Name'].fillna(0, inplace=True)
```
Other methods, go to [Missing Data](https://en.wikipedia.org/wiki/Missing_data) and have a look.

Take the **MinTemp** column as an example, and we chose the method of *impute with mean value*.

In [ ]:
# take a look at the number of null values in column MinTemp.
data_train['MinTemp'].isna().sum()

In [ ]:
# claculate the mean/median value of MinTemp
mean_temp = data_train['MinTemp'].mean()
# Then, we fill the null value with mean_temp
data_train['MinTemp']=data_train['MinTemp'].fillna(mean_temp)

Now, we check whether this step works or not.

In [ ]:
data_train['MinTemp'].isna().sum()

Yes, it's working! Now, you can follow the up step, and process other columns. Here, we provide a more effective way to handle the rest of the columns with numeric datatype.

In [ ]:
for key, values in data_train.items(): # iterate over the training data set in feature name, feature measurement pairs
    if pd.api.types.is_numeric_dtype(values): # check if numeric or continuous
        if pd.isnull(values).sum(): # if sum of null values is non-zero, the condition holds
            # Fill missing data with the mean
            data_train[key] = values.fillna(values.mean())

In [ ]:
# Print all columns with any remaining NaN values
for key, values in data_train.items():
    if pd.api.types.is_numeric_dtype(values):
        if pd.isnull(values).sum(): # if sum of null values is non-zero, the condition holds
            print(key) # print the column name

Until now, we have dealt with numeric missing values. We will move to deal with category data! The methods for handling missing data in categorical variables are quite similar with the methods for numeric variables. We can:

1. *Remove Rows with Missing Data*.
2. *Impute with Mode*: impute missing values in a categorical variable with the mode (the most frequently occurring category or randomly select from existing categories). This is a simple and often effective approach.
```python
# Impute missing values with the mode of the column
mode_value = df['Categorical_Column'].mode()[0]
df['Categorical_Column'].fillna(mode_value, inplace=True)
```
3. *Create a New Category for Missing Values*: we can create a new category label (e.g., "Missing" or "Unknown") to represent missing data in the categorical variable.


Here, we choose to fill in the missing data by randomly selecting from the existing categories. Take the **WindGustDir** as an example.

1. We first check the unique values of **WindGustDir** by using `unique` function in `pandas`.

In [ ]:
data_train["WindGustDir"].unique()

As for **WindGustDir**, only the following values exist：

'W', 'WNW', 'N', 'NNE', 'SW', 'ENE', 'SSE', 'NE', 'WSW', 'NNW', 'S', 'ESE', nan, 'NW', 'E', 'SSW', 'SE'

2. Now, we randomly select from the aforementioned values to fill in the missing data of **WindGustDir**.

In [ ]:
num_WGD=data_train['WindGustDir'].isna().sum() # number of missing values in WindGustDir
data_train.loc[data_train.WindGustDir.isna(),"WindGustDir"]=random.choices(['W','WNW','N','NNE','SW','ENE','SSE','NE','WSW','NNW','S','ESE','NW','E','SSW','SE'], k=num_WGD)

Looking at the code 
```python
data_train.loc[data_train.WindGustDir.isna(),"WindGustDir"]=random.choices(['W','WNW','N','NNE','SW','ENE','SSE','NE','WSW','NNW','S','ESE','NW','E','SSW','SE'], k=num_WGD)
```
1. `loc` is a label-based data selection method, which means that we have to pass the index of the row that we want to select. Here, we use `loc` to return the rows with missing values.
2. The `random.choices()` function generates a list of `num_WGD` random elements from the unique value list `['W','WNW','N','NNE','SW','ENE','SSE','NE','WSW','NNW','S','ESE','NW','E','SSW','SE']`, and the generated list is used to fill the missing values in the **WindGustDir** column.


The unique values array does not have to be manually typed in as just before we computed it with `unique`. We could maybe change it from
```Python
num_WGD=data_train['WindGustDir'].isna().sum()
data_train.loc[data_train.WindGustDir.isna(),"WindGustDir"]=random.choices(['W','WNW','N','NNE','SW','ENE','SSE','NE','WSW','NNW','S','ESE','NW','E','SSW','SE'], k=num_WGD)
```
to the following commented codeblock:
```Python
# Count the number of NaN values in the Wind Gust Direction column of the data
num_WGD = data_train['WindGustDir'].isna().sum()

# Lets define all possible wind directions
directions = data_train["WindGustDir"].dropna().unique() # get all unique values except for NaN

# replacce all NaN values in WindGustDir column with randomly sampled directions using random.choices function
data_train.loc[data_train.WindGustDir.isna(), "WindGustDir"] = random.choices(directions, k=num_WGD)
```

In [ ]:
data_train["WindGustDir"].isna().sum()

In [ ]:
# Check for columns which aren't numeric and with missing data
for key, values in data_train.items():
    if not pd.api.types.is_numeric_dtype(values):
        if pd.isnull(values).sum():
            print(key)

Reapeat the step 1 and 2...

In [ ]:
data_train['WindDir9am'].unique()

In [ ]:
num_WD9=data_train['WindDir9am'].isna().sum()
directions = data_train["WindDir9am"].dropna().unique() # get all unique values except for NaN
data_train.loc[data_train.WindDir9am.isna(),"WindDir9am"]=random.choices(directions, k=num_WD9)

In [ ]:
data_train['WindDir3pm'].unique()

In [ ]:
num_WD3=data_train['WindDir3pm'].isna().sum()
directions = data_train["WindDir3pm"].dropna().unique() # get all unique values except for NaN
data_train.loc[data_train.WindDir3pm.isna(),"WindDir3pm"]=random.choices(directions, k=num_WD3)

In [ ]:
data_train['RainToday'].unique()

In [ ]:
num_RT=data_train['RainToday'].isna().sum()
data_train.loc[data_train.RainToday.isna(),"RainToday"]=random.choices(['No','Yes'], k=num_RT)

In [ ]:
# check again for columns which aren't numeric and with missing data
for key, values in data_train.items():
    if not pd.api.types.is_numeric_dtype(values):
        if pd.isnull(values).sum():
            print(key)

In [ ]:
# check whether there still exists any null value in the dataset
data_train.isna().sum()

Until now, we can see that the dataset does not have missing values.

### 2.2.2 Label balancing

First, check the label distribution in current dataset!

In [ ]:
data_train['RainTomorrow'].value_counts()

In [ ]:
sns.countplot(x='RainTomorrow', data=data_train, hue='RainTomorrow')

We can find that there exists an unequal distribution of classes in the training dataset. To deal with the imbalance problem, we can do

```python
from sklearn.utils import resample
X_0 = X_sydney[(X_sydney['RainTomorrow']==0)] 
X_1 = X_sydney[(X_sydney['RainTomorrow']==1)] 

X_1_upsampled = resample(X_1, 
                         replace=True,    
                         n_samples= 1749, 
                         random_state=42) 

X_new = pd.concat([X_1_upsampled, X_0])
```


<p>Exercise: Try this code and trained a model to test it by yourself!</p>

NOTE! In a real project, we always need to balance the labels in the dataset! 


### 2.2.2 Spliting the data

Note that, to simplify this task, we only consider the data records for city **Sydney**. 

In [ ]:
X_sydney = data_train.loc[data_train['Location']=='Sydney']

In [ ]:
print(f'The number of samples in the new dataset: {len(X_sydney)}.')

In this section, we are gonna to learn how to split the data into two parts:

1. The columns of the data that we will use to make classifications, i.e., features. Denoted as **X**.
2. The column of the data that we want to predict, i.e., targets. Denoted as **Y**.

In [ ]:
X = X_sydney.drop(["RainTomorrow"], axis = 1).copy()
X.head()

In [ ]:
Y = X_sydney["RainTomorrow"].copy()
Y.head()

### 2.2.3 Formating the data

Now that we have split the dataset into `X` and `Y`, then we need to take a closer look at `X`. Let's look at the data types in `X`.

In [ ]:
X.dtypes

We see that except **Location**, **WindGustDir**, **WindDir9am**,**WindDir3pm** and **RainToday** are `object`, others are all `float64`. In order to use categorical data with skit-learn Decision Tree, we have to converts a column of categorical data into numerical values. And this operation can be accomplished via one-hot encoding or just transfer the labels with value between 0 and n_classes-1. Typically, this step can be done by `Pandas`, e.g., `get_dummies()` and `Scikit-learn`, e.g., `LabelEncoder()`. 

1. [`pd.get_dummies()`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) 

e.g.,

```python
df_test = pd.DataFrame([
    ['green','A'],
    ['red','B'],
    ['yellow','C'],
])
df_test.columns = ['color', 'label']
pd.get_dummies(df_test)
```

2. [`LabelEncoder()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html)

Here, we will give an example to use `LabelEncoder()`.

For example, Looking at the column of **Location**.

```python
X['Location'].unique()
LabelEncoder = preprocessing.LabelEncoder()
X['Location']=LabelEncoder.fit_transform(X['Location'])
X['Location'].unique()
```


In [ ]:
X['Location'].unique()

In [ ]:
LabelEncoder = preprocessing.LabelEncoder()
X['Location']=LabelEncoder.fit_transform(X['Location'])

In [ ]:
X['Location'].unique()

In [ ]:
X = X.drop(["Location"], axis = 1).copy()
X.head()

In [ ]:
# Dealing with others
X['WindGustDir']=LabelEncoder.fit_transform(X['WindGustDir'])
X['WindDir9am']=LabelEncoder.fit_transform(X['WindDir9am'])
X['WindDir3pm']=LabelEncoder.fit_transform(X['WindDir3pm'])
X['RainToday']=LabelEncoder.fit_transform(X['RainToday'])

In [ ]:
X.head()

In [ ]:
X.dtypes

In [ ]:
Y.head()

#### 2.2.4 Others

In [ ]:
# also row ID col is not important for model trainig, we remove it from X
X.drop("row ID", axis = 1, inplace = True)

Think about how to select the most imporatant features to train our model! 

In this example, we can remove **Pressure9am** and **Temp3pm**. 

Try

```python
#corelation
fig = plt.figure(figsize=[15,8])
    
sns.heatmap(X.corr(), annot=True, cmap=sns.cubehelix_palette(rot=-.4))

plt.show()
```

And what can you find from this correlation between all features?

*Since Pressure9am and Temp3pm is correlated more than 90% with other independent variable, we need tp drop them.*

```python
drop_cols = ["Pressure9am", "Temp3pm"]
X.drop(drop_cols, axis = 1, inplace = True)
```

## 3. Classification Tree

To simplify the tree, we only select the following features as inputs.
* 3. Humidity3pm - The humidity of the wind at 3 pm. (percent)
* 4. Pressure3pm - Atmospheric pressure at 3 pm. (hectopascals)
* 5. RainToday - If today is rainy then ‘Yes’. If today is not rainy then ‘No’.

In [ ]:
X_sel = X[['Humidity3pm','Pressure3pm', 'RainToday']].copy()
X_sel.head()

This notebook aims to provide brief information about how to build a classification tree based on the dataset, so we simplify this problem by removing some of the cols in the dataset.

In [ ]:
X_sel.shape

Now we simply split the data into **training** and **validation** sets and build the **Classification Tree**.

In [ ]:
# split data into training and validation sets
x_train_DT, x_test_DT, y_train_DT, y_test_DT= train_test_split(X_sel, Y, test_size=0.3, shuffle=True, random_state=42)

Now, we build up the tree ...

In [ ]:
# create a decision tree and fit it to the training data
clt = DecisionTreeClassifier()
clt = clt.fit(x_train_DT, y_train_DT)

Then we plot the tree structure.

In [ ]:
# we can plot the tree here!
plt.figure(figsize=(16,8))
plot_tree(clt,
         filled=True,
         rounded=True,
         class_names=['NO','Yes'],
         feature_names=list(X_sel.columns))
plt.show()

It's really a huge tree! And let's see how it performs on the **Testing** dataset

In [ ]:
clt.score(x_test_DT, y_test_DT)

In [ ]:
y_pred_DT = clt.predict(x_test_DT)
CM_DT = metrics.confusion_matrix(y_test_DT, y_pred_DT)
print('Confusion Matrix is\n', CM_DT)

In [ ]:
# plot the confusion matrix
sns.heatmap(CM_DT, center=True)
plt.show()

In [ ]:
# or simply use the function of scikit-learn
metrics.ConfusionMatrixDisplay.from_estimator(clt, x_test_DT, y_test_DT)
plt.show()

## 4. Optimizing - Pruning

In this section, we try to use cost complexity pruning to deal with overfitting to the training dataset, and improves the accuracy with testing dataset.

>The DecisionTreeClassifier provides parameters such as min_samples_leaf and max_depth to prevent a tree from overfiting. Cost complexity pruning provides another option to control the size of a tree. In DecisionTreeClassifier, this pruning technique is parameterized by the cost complexity parameter, ccp_alpha. Greater values of ccp_alpha increase the number of nodes pruned. Here we only show the effect of ccp_alpha on regularizing the trees and how to choose a ccp_alpha based on validation scores. (sklearn)

Ref. https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html

First, let's extract the different values of `alpha` that are available for this tree and build a pruned tree for each value for `alpha`. scikit-learn provides `DecisionTreeClassifier.cost_complexity_pruning_path` that returns the effective alphas and the corresponding total leaf impurities at each step of the pruning process.

In [ ]:
# determine values for alpha
path = clt.cost_complexity_pruning_path(x_train_DT, y_train_DT)
# extract different values for alpha
ccp_alphas, impurities = path.ccp_alphas, path.impurities

In the following plot, the maximum effective alpha value is removed, because it is the trivial tree with only one node.

In [ ]:
# exclude the maximum value for alpha
alphas = ccp_alphas[:-1] # overcome pruning all leaves.
impurities = impurities[:-1]

In [ ]:
fig, ax = plt.subplots()
ax.plot(alphas, impurities, marker="o", drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total impurity of leaves")
ax.set_title("Total Impurity vs effective alpha for training set")

In [ ]:
# create a list to store decision trees with different alpha
clts = list()
# create one decision tree for each alpha in alphas
for alpha in alphas:
    clt_ = DecisionTreeClassifier(ccp_alpha=alpha)
    clt_.fit(x_train_DT, y_train_DT)
    clts.append(clt_)

We can draw two images to show that the number of nodes and tree depth decreases as alpha increases.

In [ ]:
node_counts = [clt_.tree_.node_count for clt_ in clts]
depth = [clt_.tree_.max_depth for clt_ in clts]
fig, ax = plt.subplots(2, 1)
ax[0].plot(alphas, node_counts, marker="o", drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("number of nodes")
ax[0].set_title("Number of nodes vs alpha")
ax[1].plot(alphas, depth, marker="o", drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("depth of tree")
ax[1].set_title("Depth vs alpha")
fig.tight_layout()

Now, let's visulize the accuracy of the trees using both training and tesing datasets.

In [ ]:
print(len(alphas))
train_acc = [clt_.score(x_train_DT, y_train_DT) for clt_ in clts]
test_acc = [clt_.score(x_test_DT, y_test_DT) for clt_ in clts]

fig, ax = plt.subplots()
ax.set_xlabel('alpha')
ax.set_ylabel('accuracy')
ax.plot(alphas, train_acc, marker='o', label='train', drawstyle='steps-post')
ax.plot(alphas, test_acc, marker='o', label='test', drawstyle='steps-post')
ax.axvline(x = alphas[160], color='r')
ax.legend()
plt.show()

What can we learn from this pictures?
>When `alpha` is set to `0` and keeping the other default parameters of DecisionTreeClassifier, the tree overfits, leading to a `100%` training accuracy and `75.7%` testing accuracy. As `alpha` increases, more of the tree is pruned, thus creating a decision tree that generalizes better. 

Cross Validation for finding the best alpha!

In [ ]:
clt_1 = DecisionTreeClassifier(ccp_alpha=alphas[160])
# now we use 5-fold cross-validation creates 5 different trainig datasets
scores = cross_val_score(clt_1, x_train_DT, y_train_DT, cv=5)
df = pd.DataFrame(data={'Fold': range(5), 'Accuracy':scores})
df.plot(x='Fold', y='Accuracy', marker='o', linestyle='--')

In [ ]:
clt_1.fit(x_train_DT,y_train_DT)
clt_1.score(x_test_DT, y_test_DT)

In [ ]:
# we can plot the tree here!
plt.figure(figsize=(16,8))
plot_tree(clt_1,
         filled=True,
         rounded=True,
         class_names=['NO','Yes'],
         feature_names=list(X_sel.columns))
plt.show()

Tha alpha is sensitive to the dataset!

In [ ]:
# create a list to store the results of each fold during cross validation
alpha_loop_values = list()

# for each candidate value for alpha, we will run 5-fold cross validation,
# then we store the mean and standard deciation fo the scores for each call 
for alpha in alphas:
    clt_ = DecisionTreeClassifier(ccp_alpha=alpha)
    scores = cross_val_score(clt_, x_train_DT, y_train_DT, cv=5)
    alpha_loop_values.append([alpha, np.mean(scores), np.std(scores)])
    
alpha_results = pd.DataFrame(alpha_loop_values, columns=['alpha', 'mean', 'std'])    

alpha_results.plot(x='alpha', y='mean', yerr='std', marker='o', linestyle='--')

Using cross validation, we can find that istead of setting ccp_alpha to ..., tha alpha value close to ... is much better.

In [ ]:
alpha_results[(alpha_results['alpha']>0.0025) & (alpha_results['alpha']<0.0075)]

In [ ]:
clt_2 = DecisionTreeClassifier(ccp_alpha=alphas[161])
clt_2.fit(x_train_DT,y_train_DT)
clt_2.score(x_test_DT, y_test_DT)

In [ ]:
# we can plot the tree here!
plt.figure(figsize=(16,8))
plot_tree(clt_2,
         filled=True,
         rounded=True,
         class_names=['NO','Yes'],
         feature_names=list(X_sel.columns))

In [ ]:
metrics.ConfusionMatrixDisplay.from_estimator(clt_1, x_test_DT, y_test_DT)
plt.show()

In [ ]:
metrics.ConfusionMatrixDisplay.from_estimator(clt_2, x_test_DT, y_test_DT)
plt.show()

## 4. Random forest

Random forest is a commonly-used machine learning algorithm trademarked by Leo Breiman and Adele Cutler, which combines the output of multiple decision trees to reach a single result. Its ease of use and flexibility have fueled its adoption, as it handles both classification and regression problems. 

Ref. https://www.ibm.com/topics/random-forest

### 4.1 Create Random Forest

Use `RandomForestClassifier` from `sklearn.ensemble` to create a random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
features_name = list(X.columns)
features_name

In [ ]:
# split data into training and validation sets
x_train, x_test, y_train, y_test= train_test_split(X, Y, test_size=0.3, shuffle=True, random_state=42)

In [ ]:
# create a random forest
rand_forest = RandomForestClassifier(random_state=42)
rand_forest.fit(x_train, y_train)

In [ ]:
acc = rand_forest.score(x_test, y_test)
print(f'Random Forest Test Score is : {acc}')

In [ ]:
y_pred_RF = rand_forest.predict(x_test)
CM_RF = metrics.confusion_matrix(y_test, y_pred_RF)

sns.heatmap(CM_RF, center=True)
plt.show()

print('Confusion Matrix is\n', CM_RF)


## 5. GBDT

Gradient-boosted decision trees are a machine learning technique for optimizing the predictive value of a model through successive steps in the learning process. Each iteration of the decision tree involves adjusting the values of the coefficients, weights, or biases applied to each of the input variables being used to predict the target value, with the goal of minimizing the loss function (the measure of difference between the predicted and actual target values). The gradient is the incremental adjustment made in each step of the process; boosting is a method of accelerating the improvement in predictive accuracy to a sufficiently optimum value.

Ref. https://c3.ai/glossary/data-science/gradient-boosted-decision-trees-gbdt/

We use `GradientBoostingClassifier` from `sklearn.ensemble` to create a GBDT.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
GBDTModel = GradientBoostingClassifier(n_estimators=200, max_depth=11, learning_rate=0.07, random_state=44)
GBDTModel.fit(x_train, y_train)

Please change the parameters and see what happens!
For example, change the learning rate!

In [ ]:
acc = GBDTModel.score(x_test, y_test)
print(f'GBDTModel Test Score is : {acc}')

In [ ]:
y_pred_GBDT = GBDTModel.predict(x_test)
CM_GBDT = metrics.confusion_matrix(y_test, y_pred_GBDT)

sns.heatmap(CM_GBDT, center=True)
plt.show()

print('Confusion Matrix is\n', CM_GBDT)

## 6 Feature Importance

Feature importance in tree-based models refers to a measure that quantifies the contribution of each feature (or variable) in the model's decision-making process. It provides insights into which features have the most significant impact on the model's predictions. Feature importance is particularly relevant in tree-based models like Decision Trees, Random Forests, and Gradient Boosted Trees. 

### 6.1 Feature importance based on mean decress in imputiy (MDI)

Ref. https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html

Feature importances are provided by the fitted attribute `feature_importances_` and they are computed as the mean and standard deviation of accumulation of the impurity decrease within each tree.

In [ ]:
# We take random forest as an example.
importances = rand_forest.feature_importances_
std = np.std([tree.feature_importances_ for tree in rand_forest.estimators_], axis=0)
forest_importances = pd.Series(importances, index=x_train.columns)

In [ ]:
fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

### 6.2 Feature importance based on feature permutation

Permutation feature importance overcomes limitations of the impurity-based feature importance: they do not have a bias toward high-cardinality features and can be computed on a left-out test set.

In [ ]:
from sklearn.inspection import permutation_importance
result = permutation_importance(
    rand_forest, x_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)
forest_importances = pd.Series(result.importances_mean, index=x_train.columns)

In [ ]:
fig, ax = plt.subplots()
forest_importances.plot.bar(yerr=result.importances_std, ax=ax)
ax.set_title("Feature importances using permutation on full model")
ax.set_ylabel("Mean accuracy decrease")
fig.tight_layout()

### 6.3 Feature Selection Using Tree-Based (or Gini) Importance

The classes in the `sklearn.feature_selection` module can be used for feature selection/dimensionality reduction on sample sets, either to improve estimators’ accuracy scores or to boost their performance on very high-dimensional datasets.


Ref. https://scikit-learn.org/stable/modules/feature_selection.html

#### 6.3.1 Using the Feature Importance Values

In [ ]:
feature_importance = GBDTModel.feature_importances_
importance_df = pd.DataFrame({'features': x_train.columns,
                              'importance': feature_importance})
importance_df.sort_values(by='importance', ascending=False, inplace=True)
importance_df

Create a list of the features with Gini importance greater than 0.04 and use that list to retrain the model.

In [ ]:
feature_list = importance_df[importance_df.importance > 0.04]['features'].tolist()
feature_list

In [ ]:
# Here, we keep the top 5 features for training
feature_list = importance_df['features'].head(5).tolist()
feature_list

In [ ]:
x_train_new = x_train[feature_list]
x_test_new = x_test[feature_list]

In [ ]:
reduced_GBDT = GradientBoostingClassifier(n_estimators=200, max_depth=11, learning_rate=0.07, random_state=44)
reduced_GBDT.fit(x_train_new, y_train)

In [ ]:
acc = reduced_GBDT.score(x_test_new, y_test)
print(f'After feature selecting, GBDTModel Test Score is : {acc}')

In [ ]:
y_pred_RDGBDT = reduced_GBDT.predict(x_test_new)
CM_RDGBDT = metrics.confusion_matrix(y_test, y_pred_RDGBDT)

sns.heatmap(CM_RDGBDT, center=True)
plt.show()

print('Confusion Matrix is\n', CM_RDGBDT)

#### 6.3.2 Using SelectFromModel

Use SelectFromModel to use model.feature_importances_ to pick the features to remove. Since the model has already been fit, pass prefit=True to use the prefit model. As Origin has outsized Gini importance, set a low threshold with threshold='0.01*mean' to keep multiple features.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(GBDTModel, prefit=True, threshold='0.01*mean')
x_train_new2 = selector.transform(x_train)
x_test_new2 = selector.transform(x_test)
print(x_train.shape, x_train_new2.shape)

In [ ]:
reduced_model = GradientBoostingClassifier(n_estimators=200, max_depth=11, learning_rate=0.07, random_state=44)
reduced_model.fit(x_train_new2, y_train)

In [ ]:
acc = reduced_model.score(x_test_new2, y_test)
print(f'After feature selecting, GBDTModel Test Score is : {acc}')

In [ ]:
y_pred_RDGBDT2 = reduced_model.predict(x_test_new2)
CM_RDGBDT2 = metrics.confusion_matrix(y_test, y_pred_RDGBDT)

sns.heatmap(CM_RDGBDT2, center=True)
plt.show()

print('Confusion Matrix is\n', CM_RDGBDT2)

## 7 Exercise

1. Try the following ensemble methods by yourself:
- XGBoost https://xgboost.readthedocs.io/en/latest/index.html
- Light https://lightgbm.readthedocs.io/en/latest/index.html
- CatBoost https://catboost.ai/en/docs/

2. Try to balance the dataset, and train the models again.